In [ ]:
!apt-get update
!apt-get install g++ openjdk-8-jdk
!pip3 install konlpy

In [ ]:
from google.colab import files
myfile = files.upload()

In [ ]:
!pip install gdown
!gdown --id 15kGgL0GGkzZdUzlJZwayDzHA7HZ9hoqW --output /content/newsData.zip

In [ ]:
!unzip -qq '/content/newsData.zip' -d '/content/'

In [ ]:
# 데이터 카테고리
# 0: 정치 (policy)
# 1: 경제 (economy)
# 2: 사회 (social)
# 3: 생활/문화 (culture)
# 4: 세계 (international)
# 5: 과학 (science)
# 6: 연애 (ENT)
# 7. 스포츠 (sports)

In [ ]:
import os
import pandas as pd
import re

folders = ["/content/0", "/content/1", "/content/2", "/content/3",
           "/content/4", "/content/5", "/content/6", "/content/7"]

data_list = []

def remove_text(text):
    return re.sub(r'[^a-zA-Z0-9가-힣\s]', '', text)

for folder in folders:
  label = int(os.path.basename(folder))
  for filename in os.listdir(folder):
      file_path = os.path.join(folder, filename)
      if os.path.isfile(file_path) and filename.endswith('.txt'):
          with open(file_path, 'r', encoding='utf-8') as file:
              content = file.read().strip()
              content = content.replace('\n', ' ').replace('\t', ' ')
              content = remove_text(content)
              data_list.append({'data': content, 'label': label})

df = pd.DataFrame(data_list)

df.head(5)

In [ ]:
selected_files = ['0188NewsData.txt', '0108NewsData.txt', '0186NewsData.txt']

print(selected_files)

In [ ]:
# 파일 읽기
f = open('/content/0/0186NewsData.txt', 'r', encoding='utf-8')
data = f.read()
print(data)

In [ ]:
df.shape

In [ ]:
filtered_df = df[df['data'].str.contains('드루킹')]

filtered_df.head(5)

In [ ]:
filtered_df = df[df['data'].str.contains('경찰 구치소 접견조사')]

print(filtered_df)

In [ ]:
row_num = [140,72,112,149,57]

new_df = df.iloc[row_num]
new_df = new_df.reset_index(drop=True)

new_df

In [ ]:
k_stopword = pd.read_csv('/content/korean_stopword.csv')

stopword = list(k_stopword['불용어'])+['을', '은', '를', '이가', '과', '의', '는', '에']
stopword[:5]

In [ ]:
stopword = list(set(stopword))

In [ ]:
import pandas as pd
import urllib.request
import matplotlib.pyplot as plt
import re
from konlpy.tag import Okt
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np

okt = Okt()

In [ ]:
from tqdm.notebook import tqdm

X_train = []
for i in tqdm(df.index):
  morph = okt.morphs(df.loc[i]['data'], stem=True) # stem=True : 어근 찾아오기
  temp_X = []
  for txt in morph:
    if txt not in stopword:
      temp_X.append(txt)
  X_train.append(temp_X)
X_train[:5]

In [ ]:
df['label'].value_counts().plot(kind = 'bar')

In [ ]:
df.info()

In [ ]:
df = df.dropna(how = 'any')
print('전체 데이터 프레임:', df.isnull().values.any())

In [ ]:
y= df['label']

In [ ]:
train_data = df.sample(frac=0.8, random_state=1)
test_data = df.drop(train_data.index)

In [ ]:
print('학습(train) 데이터의 수:', len(train_data))

In [ ]:
print('학습(test) 데이터의 수 : ', len(test_data))

In [ ]:
print('### 학습데이터의 라벨 분포 ###\n', train_data['label'].value_counts())

In [ ]:
print('### 테스트데이터의 라벨 분포 ###\n', test_data['label'].value_counts())

In [ ]:
X_train = []
for i in tqdm(train_data.index):
  morph = okt.morphs(train_data.loc[i]['data'], stem=True) # stem=True : 어근 찾아오기
  temp_X = []
  for txt in morph:
    if txt not in stopword:
      temp_X.append(txt)
  X_train.append(temp_X)
X_train[:5]

In [ ]:
X_test = []
for i in tqdm(test_data.index):
  morph = okt.morphs(test_data.loc[i]['data'], stem=True) # stem=True : 어근 찾아오기
  temp_X = []
  for txt in morph:
    if txt not in stopword:
      temp_X.append(txt)
  X_test.append(temp_X)
X_test[:5]

In [ ]:
drop_train = []

for index, sentence in enumerate(X_train):
  if len(sentence) < 1:
    drop_train.append(index)
drop_train[:3]

In [ ]:
len(drop_train)

In [ ]:
X_train = np.array(X_train, dtype=object)
X_train

y_train = np.array(train_data['label'])
y_train[:3]

In [ ]:
X_train = np.delete(X_train, drop_train, axis=0)
y_train = np.delete(y_train, drop_train, axis=0)
print(len(X_train))
print(len(y_train))

In [ ]:
drop_test = []

for index, sentence in enumerate(X_test):
  if len(sentence) < 1:
    drop_test.append(index)
drop_test[:3]

In [ ]:
X_test = np.array(X_test, dtype=object)
y_test = np.array(test_data['label'])

In [ ]:
X_test = np.delete(X_test, drop_test, axis=0)
y_test = np.delete(y_test, drop_test, axis=0)
print(len(X_test))
print(len(y_test))

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)
print(tokenizer.word_index)

In [ ]:
threshold = 3
total_cnt = len(tokenizer.word_index)
rare_cnt = 0
total_freq = 0
rare_freq = 0

for key, value in tokenizer.word_counts.items():
    total_freq = total_freq + value
    if(value < threshold):
        rare_cnt = rare_cnt + 1
        rare_freq = rare_freq + value

print('단어 집합(vocabulary)의 크기:', total_cnt)
print(f'등장 빈도가 {threshold - 1}번 이하인 희귀 단어의 수:{rare_cnt}')
print(f'단어 집합에서 희귀 단어의 비율:{(rare_cnt / total_cnt)*100:.3f}')
print(f'전체 등장 빈도에서 희귀 단어 등장 빈도 비율:{(rare_freq / total_freq)*100:.3f}')

In [ ]:
vocab_size = total_cnt - rare_cnt + 2
print('단어 집합의 크기:', vocab_size)

In [ ]:
tokenizer = Tokenizer(vocab_size, oov_token='OOV')
tokenizer.fit_on_texts(X_train)
X_train = tokenizer.texts_to_sequences(X_train)
X_train[:3]

print(tokenizer.word_index)

In [ ]:
print('문서의 최대 길이:', max(len(l) for l in X_train))
print('문서의 평균 길이:', sum(map(len, X_train)) / len(X_train))
plt.hist([len(s) for s in X_train], bins=50)
plt.xlabel('length of samples')
plt.ylabel('number of samples')
plt.show()

In [ ]:
def below_threshold_len(max_len, nested_list):
  cnt = 0
  for s in nested_list:
    if(len(s) <= max_len):
      cnt = cnt + 1
  print(f'전체 샘플 중 길이가 {max_len} 이하인 샘플의 비율: {(cnt / len(nested_list))*100:.3f}')

In [ ]:
max_len = 500
below_threshold_len(max_len, X_train)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train = pad_sequences(X_train, maxlen=max_len)

X_test = tokenizer.texts_to_sequences(X_test)
X_test = pad_sequences(X_test, maxlen=max_len)
print('X_train 크기:', X_train.shape)
print('X_test 크기:', X_test.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Dense, LSTM, Embedding, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model_LSTM = Sequential()
model_LSTM.add(Embedding(input_dim = vocab_size, output_dim = 64))
model_LSTM.add(LSTM(64, return_sequences = True))
model_LSTM.add(LSTM(32, return_sequences = False))
model_LSTM.add(Dense(32, activation = 'relu'))
model_LSTM.add(Dense(8, activation = 'softmax'))

In [ ]:
model_LSTM.compile(optimizer = 'adam',
                  loss = 'sparse_categorical_crossentropy',
                  metrics = ['acc'])

In [ ]:
model_LSTM.summary()

In [ ]:
es = EarlyStopping(monitor = 'val_loss', mode = 'min', verbose = 1, patience = 4)
mc = ModelCheckpoint('best_model_LSTM.keras', monitor = 'val_acc', mode = 'max', verbose = 2, save_best_only = True)

history = model_LSTM.fit(X_train, y_train, epochs = 30, callbacks=[es, mc] , batch_size= 60, validation_split=0.2)

In [ ]:
import matplotlib.pyplot as plt

his_dict = history.history
loss = his_dict['loss']
val_loss = his_dict['val_loss']

epochs = range(1, len(loss) + 1)
fig = plt.figure(figsize=(10, 5))

# 훈련 및 검증 손실 그리기
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(epochs, loss, color = 'blue', label='train_loss')
ax1.plot(epochs, val_loss, color = 'orange', label='val_loss')
ax1.set_title('train and val loss')
ax1.set_xlabel('epochs')
ax1.set_ylabel('loss')
ax1.legend()

acc = his_dict['acc']
val_acc = his_dict['val_acc']

ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(epochs, acc, color = 'blue', label='train_acc')
ax2.plot(epochs, val_acc, color = 'orange', label='val_acc')
ax2.set_title('train and val acc')
ax2.set_xlabel('epochs')
ax2.set_ylabel('acc')
ax2.legend()

plt.show()

In [ ]:
print('테스트 정확도:', model_LSTM.evaluate(X_test, y_test)[1])

In [ ]:
loaded_model_LSTM= load_model('best_model_LSTM.keras')

In [ ]:
new_sentence = input('리뷰입력:')
new_sentence = okt.morphs(new_sentence)
new_sentence

In [ ]:
new_sentence = [word for word in new_sentence if not word in stopword] # 불용어 제거
new_sentence

In [ ]:
encoded = tokenizer.texts_to_sequences(new_sentence) # 정수 인코딩
encoded

In [ ]:
pad_new = sequence.pad_sequences(encoded, maxlen=max_len) # 패딩
pad_new

In [ ]:
score = float(sentiment_predict_LSTM.predict(pad_new))

In [ ]:
if (score < 1):
  print('해당 기사의 테마는 정치입니다.')
elif (score < 2):
  print('해당 기사의 테마는 경제입니다.')
elif (score < 3):
  print('해당 기사의 테마는 사회입니다.')
elif (score < 4):
  print('해당 기사의 테마는 생활/문화입니다.')
elif (score < 5):
 print('해당 기사의 테마는 세계입니다.')
elif (score < 6):
  print('해당 기사의 테마는 과학입니다.')
elif (score < 7):
  print('해당 기사의 테마는 연애입니다.')
elif (score < 8):
  print('해당 기사의 테마는 스포츠입니다.')
else:
  print('기사 없음')

In [ ]:
from tensorflow.keras.models import load_model
loaded_model_LSTM = load_model('best_model_LSTM.keras')

def sentiment_predict_LSTM(new_sentence):
  new_sentence = okt.morphs(new_sentence) # 토큰화
  new_sentence = [word for word in new_sentence if not word in stopword]
  encoded = tokenizer.texts_to_sequences([new_sentence]) # 정수 인코딩
  pad_new = sequence.pad_sequences(encoded, maxlen=max_len) # 패딩
  score = float(loaded_model_LSTM.predict(pad_new))
  if (score < 1):
    print('해당 기사의 테마는 정치입니다.')
  elif (score < 2):
    print('해당 기사의 테마는 경제입니다.')
  elif (score < 3):
    print('해당 기사의 테마는 사회입니다.')
  elif (score < 4):
    print('해당 기사의 테마는 생활/문화입니다.')
  elif (score < 5):
    print('해당 기사의 테마는 세계입니다.')
  elif (score < 6):
    print('해당 기사의 테마는 과학입니다.')
  elif (score < 7):
    print('해당 기사의 테마는 연애입니다.')
  elif (score < 8):
    print('해당 기사의 테마는 스포츠입니다.')
  else:
    print('기사 없음')

In [ ]:
sentiment_predict_LSTM('이정도면 많이 했다...졌지만 잘싸웠다')

In [ ]:
sentiment_predict_LSTM()